<a href="https://colab.research.google.com/github/DaniloDuque/logistic-regression/blob/main/src/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Regresión Logística

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')

sys.path.insert(0, os.path.abspath('src'))

---
# Sección 1 — Algoritmo de Regresión Logística

## 1.a — Usar el código del perceptrón como base
DOCUMENTACION EN LATEX: Explique en el reporte cada una de las modificaciones necesarias al código del perceptrón, utilizando como referencia las diferencias entre ambos algoritmos.

### Pruebas unitarias
2 pruebas unitarias (con su objetivo, entradas, salidas esperadas y resultados) para las funciones modificadas en el algoritmo del perceptron base.

---
## 1.b — Experimentos: Datos separables vs no separables

### Imports

In [ ]:
# Install spacy
!pip install spacy

# Download the Spanish model
!python -m spacy download es_core_news_lg

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import spacy
from pathlib import Path

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_single_result, print_runs_table, compute_accuracy
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

FIGURES_DIR = Path(__file__).parent.parent / 'figures' if '__file__' in dir() else Path('..') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)


### Experimento 1 — Datos linealmente separables

Dos clases generadas con `cluster_std=1.0` (grupos compactos y bien separados).
Se espera que el modelo converja rápidamente y obtenga un MAE bajo.


In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
train_errors_s, test_errors_s = train_with_history(
    model_s, X_train_s, y_train_s, X_test_s, y_test_s, steps=STEPS, alpha=ALPHA
)

mae_train_s = compute_mae(y_train_s, model_s.forward(X_train_s))
mae_test_s  = compute_mae(y_test_s,  model_s.forward(X_test_s))
acc_train_s = compute_accuracy(model_s, X_train_s, y_train_s)
acc_test_s  = compute_accuracy(model_s, X_test_s,  y_test_s)

print(f"Iteraciones: {STEPS}")
print_single_result('Linealmente separable', mae_train_s, mae_test_s, acc_train_s, acc_test_s)

plot_results(model_s, X_train_s, y_train_s, train_errors_s, test_errors_s,
             'Experimento 1 — Datos linealmente separables',
             output_path=FIGURES_DIR / 'experimento1.pdf')


### Experimento 2 — Datos no linealmente separables

Dos clases generadas con `cluster_std=4.0` (grupos solapados).
Se espera que el modelo tenga dificultades y obtenga un MAE más alto.


In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
train_errors_ns, test_errors_ns = train_with_history(
    model_ns, X_train_ns, y_train_ns, X_test_ns, y_test_ns, steps=STEPS, alpha=ALPHA
)

mae_train_ns = compute_mae(y_train_ns, model_ns.forward(X_train_ns))
mae_test_ns  = compute_mae(y_test_ns,  model_ns.forward(X_test_ns))
acc_train_ns = compute_accuracy(model_ns, X_train_ns, y_train_ns)
acc_test_ns  = compute_accuracy(model_ns, X_test_ns,  y_test_ns)

print(f"Iteraciones: {STEPS}")
print_single_result('No linealmente separable', mae_train_ns, mae_test_ns, acc_train_ns, acc_test_ns)

plot_results(model_ns, X_train_ns, y_train_ns, train_errors_ns, test_errors_ns,
             'Experimento 2 — Datos no linealmente separables',
             output_path=FIGURES_DIR / 'experimento2.pdf')


### 1.2 — 10 ejecuciones: MAE medio, precisión, mínimo y máximo


In [ ]:
results_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
results_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)

print_runs_table(results_s, results_ns)

---
# Sección 2 — Regresión Logística y LLMs para clasificación de textos

## Pre requisitos


In [ ]:
# Prerequisitos
import torch

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from dataset import load_feina
from logistic_regression import LogisticRegression

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

#!pip install spacy
#!python -m spacy download es_core_news_lg
#import spacy

#!pip install simplemma
#import simplemma

# Cargar dataset de FEINA
df, all_texts, all_labels = load_feina()
#    df         : pd.DataFrame  — dataframe completo
#    texts      : list[str]     — textos (Segment + Proposal concatenados)
#    labels     : list[int]     — 1 = complejo (Segment), 0 = simple (Proposal)


STEPS_LR = 3000


#Sección 2.1 - Regresión logística con la TFIDF.

In [ ]:
# Preprocesamiento del texto

def preprocess_text(document):
    tokeniser = RegexpTokenizer(r'\w+')
    tokens = tokeniser.tokenize(document)

    lemmatiser = WordNetLemmatizer()
    lemmas = [lemmatiser.lemmatize(token.lower(), pos='v') for token in tokens]

    keywords = [lemma for lemma in lemmas if lemma not in stopwords.words('spanish')]
    return keywords

print("── a) Examen cualitativo del preprocesamiento ──")
for text in all_texts[:3]:
    print(f"  ORIGINAL:  {text}")
    print(f"  PROCESADO: {preprocess_text(text)}")
    print()

In [ ]:
# Construcción de descriptores TF-IDF
def display_tfidfs(X_vectorised):
    df_sparse = pd.DataFrame.sparse.from_spmatrix(X_vectorised)
    print(df_sparse)

tfidf_vectoriser = TfidfVectorizer(analyzer=preprocess_text)
corpus_df = pd.DataFrame({'corpus': all_texts})
X_train_vectorised = tfidf_vectoriser.fit_transform(corpus_df['corpus'])

# Examen cualitativo con textos de prueba elegidos del propio dataset
print("── b) Examen cualitativo de descriptores TF-IDF ──")
test_samples = all_texts[6000:6003]
test_vectorised = tfidf_vectoriser.transform(test_samples)
display_tfidfs(test_vectorised)

In [ ]:
import pandas as pd
import torch
# Transformación a matriz
X_tfidf = pd.DataFrame.sparse.from_spmatrix(X_train_vectorised)
print(f"\n── c) Forma de la matriz TF-IDF: {X_tfidf.shape} ──")  # (N, vocab_size)


X = torch.tensor(X_tfidf.values, dtype=torch.float64)
t = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)

In [ ]:
# Clasificación con regresión logística

X_train, X_test, t_train, t_test = train_test_split(
    X, t, test_size=0.2, random_state=42
)

model_tfidf = LogisticRegression(
    torch.zeros((X.shape[1], 1), dtype=torch.float64)
)
model_tfidf.train(X_train, t_train, steps=STEPS_LR)


In [ ]:
print("\n── d) Resultados ──")
print(f"  Train accuracy: {model_tfidf.accuracy(X_train, t_train):.4f}")
print(f"  Test  accuracy: {model_tfidf.accuracy(X_test,  t_test):.4f}")


#Sección 2.2 - Regresión logística con vectores empotrados de BERT.

### a) Transformar entrada en vectores empotrados


In [ ]:



from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"


# ── Modelo 1: BERT en español (dccuchile) ─────────────────────────
tokenizerBERT = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
modelBERT = AutoModelForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
modelBERT.eval()

# ── Modelo 2: BERTIN - RoBERTa en español ─────────────────────────
tokenizerRoBERTa = AutoTokenizer.from_pretrained("bertin-project/bertin-roberta-base-spanish")
modelRoBERTa = AutoModelForMaskedLM.from_pretrained("bertin-project/bertin-roberta-base-spanish")
modelRoBERTa.eval()

# ── Embedding mediante token CLS ──────────────────────────────────
def get_embeddings_batch(texts, model, tokenizer, batch_size=32, device="cpu"):
    model = model.to(device)
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True      # padea al más largo del batch
        ).to(device)

        with torch.no_grad():
            outputs = model.base_model(**inputs)

        # Token CLS de cada elemento del batch → shape (batch_size, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0).numpy()  # (N, 768)


embInputsBERT    = get_embeddings_batch(all_texts, modelBERT,    tokenizerBERT,    device=device)
embInputsRoBERTa = get_embeddings_batch(all_texts, modelRoBERTa, tokenizerRoBERTa, device=device)

### b) Entrenar regresor logístico con el dataset empotrado

In [ ]:
from logistic_regression import LogisticRegression

STEPS_LR = 10000

#Con BERT
X = torch.tensor(embInputsBERT, dtype=torch.float64)
t = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)  # (N,) → (N, 1)

modelB = LogisticRegression(torch.zeros((X.shape[1], 1), dtype=torch.float64))
modelB.train(X, t, STEPS_LR)

#Con RoBERTa
M = torch.tensor(embInputsRoBERTa, dtype=torch.float64)
y = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)  # (N,) → (N, 1)

modelR = LogisticRegression(torch.zeros((X.shape[1], 1), dtype=torch.float64))
modelR.train(M, y, STEPS_LR)

### c) Evaluar con los mismos datos de entrenamiento la clasificación que hace entrenado

In [ ]:
accB = modelB.accuracy(X, t)
print(f"Accuracy BERT: {accB}")

accR = modelR.accuracy(M, y)
print(f"Accuracy RoBERTa: {accR}")


Modelo                Config                     Acc     MAE

selectra              1shot_newline           0.4999  0.5001
selectra              3shot_newline           0.5039  0.4961
selectra              5shot_newline           0.4894  0.5106
bart                  1shot_newline           0.4992  0.5008
bart                  3shot_newline           0.5003  0.4997
bart                  5shot_newline           0.5000  0.5000


---
# Sección 2.5 — Prueba completa: 30 corridas con partición 80/20

Se implementa la prueba completa de todos los tratamientos definidos en las secciones anteriores:
- **TF-IDF + Regresión Logística**
- **BERT + Regresión Logística**
- **RoBERTa + Regresión Logística**
- **LLM Selectra (zero-shot)**
- **LLM BART (zero-shot)**
- **Few-Shot 1-shot (Selectra)**
- **Few-Shot 3-shot (Selectra)**
- **Few-Shot 5-shot (Selectra)**

Se realizan **30 corridas** con particiones **80% entrenamiento / 20% prueba** aleatorias.
La misma partición se usa para todos los tratamientos en cada corrida.
Se reportan medias y desviaciones estándar de la accuracy.

### Pre-requisito: matrices precalculadas

Esta sección asume que los embeddings y la matriz TF-IDF ya fueron calculados
en las secciones anteriores y están disponibles como:
- `embInputsBERT` — array numpy (N, 768)
- `embInputsRoBERTa` — array numpy (N, 768)
- `X_tfidf_full` — array numpy (N, vocab_size) con el TF-IDF completo
- `all_texts`, `all_labels` — del dataset FEINA

La celda siguiente extrae `X_tfidf_full` desde el vectorizador ya entrenado.

In [ ]:
# Obtener la matriz TF-IDF completa como numpy array (precalculada en 2.1)
X_tfidf_full = tfidf_vectoriser.transform(
    pd.DataFrame({'corpus': all_texts})['corpus']
).toarray()  # (N, vocab_size)

print(f"X_tfidf_full shape:    {X_tfidf_full.shape}")
print(f"embInputsBERT shape:   {embInputsBERT.shape}")
print(f"embInputsRoBERTa shape:{embInputsRoBERTa.shape}")
print(f"all_labels length:     {len(all_labels)}")


### Resultados — Medias y desviaciones estándar

La siguiente celda compila e imprime la tabla de resultados de las 30 corridas.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Sección 2.5 — 30 corridas con partición 80/20
# ══════════════════════════════════════════════════════════════════

import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from logistic_regression import LogisticRegression
from llm_classifier import load_classifiers, compute_llm_metrics
from few_shot_classifier import (
    load_few_shot_classifiers, classify_few_shot_batch, MODELS, CONFIGS
)

N_RUNS    = 30
TEST_SIZE = 0.2
STEPS_LR  = 1000

# ── Nombres de tratamientos ───────────────────────────────────────
TREATMENTS = [
    "TF-IDF + LogReg",
    "BERT + LogReg",
    "RoBERTa + LogReg",
    "Selectra (zero-shot)",
    "BART (zero-shot)",
    "Selectra 1-shot",
    "Selectra 3-shot",
    "Selectra 5-shot",
]

# Resultados acumulados: dict {treatment: list of accuracies}
results_30 = {t: [] for t in TREATMENTS}

# ── Cargar LLMs una sola vez (fuera del loop) ─────────────────────
print("Cargando clasificadores LLM...")
zeroshot_clfs  = load_classifiers()
fewshot_clfs   = load_few_shot_classifiers()
print("Modelos cargados.")

y_all = np.array(all_labels)

# ── Loop de 30 corridas ───────────────────────────────────────────
for run in range(N_RUNS):
    print(f"\n── Corrida {run + 1}/{N_RUNS} ──")

    # Índices de la partición — MISMA para todos los tratamientos
    idx = np.arange(len(all_texts))
    idx_train, idx_test = train_test_split(
        idx, test_size=TEST_SIZE, random_state=run  # random_state=run garantiza reproducibilidad
    )

    y_train = y_all[idx_train]
    y_test  = y_all[idx_test]

    # ── 1. TF-IDF + Regresión Logística ──────────────────────────
    X_tfidf_train = X_tfidf_full[idx_train]
    X_tfidf_test  = X_tfidf_full[idx_test]

    t_train_tf = torch.tensor(y_train, dtype=torch.float64).unsqueeze(1)
    t_test_tf  = torch.tensor(y_test,  dtype=torch.float64).unsqueeze(1)

    m_tfidf = LogisticRegression(
        torch.zeros((X_tfidf_train.shape[1], 1), dtype=torch.float64)
    )
    m_tfidf.train(
        torch.tensor(X_tfidf_train, dtype=torch.float64),
        t_train_tf, steps=STEPS_LR
    )
    acc = m_tfidf.accuracy(
        torch.tensor(X_tfidf_test, dtype=torch.float64), t_test_tf
    )
    results_30["TF-IDF + LogReg"].append(float(acc))

    # ── 2. BERT + Regresión Logística ────────────────────────────
    emb_bert_train = embInputsBERT[idx_train]
    emb_bert_test  = embInputsBERT[idx_test]

    t_train_b = torch.tensor(y_train, dtype=torch.float64).unsqueeze(1)
    t_test_b  = torch.tensor(y_test,  dtype=torch.float64).unsqueeze(1)

    m_bert = LogisticRegression(
        torch.zeros((emb_bert_train.shape[1], 1), dtype=torch.float64)
    )
    m_bert.train(
        torch.tensor(emb_bert_train, dtype=torch.float64),
        t_train_b, steps=STEPS_LR
    )
    acc = m_bert.accuracy(
        torch.tensor(emb_bert_test, dtype=torch.float64), t_test_b
    )
    results_30["BERT + LogReg"].append(float(acc))

    # ── 3. RoBERTa + Regresión Logística ─────────────────────────
    emb_rob_train = embInputsRoBERTa[idx_train]
    emb_rob_test  = embInputsRoBERTa[idx_test]

    t_train_r = torch.tensor(y_train, dtype=torch.float64).unsqueeze(1)
    t_test_r  = torch.tensor(y_test,  dtype=torch.float64).unsqueeze(1)

    m_rob = LogisticRegression(
        torch.zeros((emb_rob_train.shape[1], 1), dtype=torch.float64)
    )
    m_rob.train(
        torch.tensor(emb_rob_train, dtype=torch.float64),
        t_train_r, steps=STEPS_LR
    )
    acc = m_rob.accuracy(
        torch.tensor(emb_rob_test, dtype=torch.float64), t_test_r
    )
    results_30["RoBERTa + LogReg"].append(float(acc))

    # ── 4 & 5. LLMs zero-shot ────────────────────────────────────
    # Las predicciones zero-shot son deterministas (no dependen del split de train),
    # solo varía el subconjunto de test evaluado.
    texts_test = [all_texts[i] for i in idx_test]

    from llm_classifier import classify_batch, MODELS as ZMODELS
    for model_key, treatment_key in [("selectra", "Selectra (zero-shot)"),
                                      ("bart",     "BART (zero-shot)")]:
        cfg   = ZMODELS[model_key]
        preds = classify_batch(
            zeroshot_clfs[model_key],
            texts_test,
            cfg["labels"],
            cfg["template"],
        )
        acc = (preds == y_test).mean()
        results_30[treatment_key].append(float(acc))

    # ── 6, 7, 8. Few-shot (Selectra, distintos n_shots) ──────────
    for config_key, treatment_key in [
        ("1shot_newline", "Selectra 1-shot"),
        ("3shot_newline", "Selectra 3-shot"),
        ("5shot_newline", "Selectra 5-shot"),
    ]:
        preds = classify_few_shot_batch(
            fewshot_clfs["selectra"],
            texts_test,
            model_key="selectra",
            config_key=config_key,
        )
        acc = (preds == y_test).mean()
        results_30[treatment_key].append(float(acc))

print("\n✓ 30 corridas completadas.")


### Resultados — Medias y desviaciones estándar

La siguiente celda compila e imprime la tabla de resultados de las 30 corridas.

In [ ]:
# ── Tabla de resultados ──────────────────────────────────────────
print("\n" + "=" * 62)
print(f"{'Tratamiento':<28s}  {'Media Acc':>9s}  {'Std Acc':>8s}")
print("=" * 62)

summary = {}
for treatment, accs in results_30.items():
    mean = np.mean(accs)
    std  = np.std(accs)
    summary[treatment] = {"mean": mean, "std": std}
    print(f"{treatment:<28s}  {mean:>9.4f}  {std:>8.4f}")

print("=" * 62)


In [ ]:
# ── Gráfico de barras con error bars ─────────────────────────────
import matplotlib.pyplot as plt

labels  = list(summary.keys())
means   = [summary[t]["mean"] for t in labels]
stds    = [summary[t]["std"]  for t in labels]

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(x, means, yerr=stds, capsize=5,
              color='steelblue', alpha=0.8, ecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=10)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy media (30 corridas, 80/20) — todos los tratamientos")
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Baseline aleatorio')
ax.legend()
plt.tight_layout()
plt.savefig("figures/sec5_30runs_accuracy.png", dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────────
# Sección 2.3 — Modelos LLM para clasificación de complejidad
# Modelos:
#   1. Recognai/zeroshot_selectra_medium  (español, ELECTRA-based)
#   2. facebook/bart-large-mnli           (multilingüe, BART-based)
# ─────────────────────────────────────────────────────────────────

In [ ]:
# ── Sección 2.3 — LLMs ───────────────────────────────────────────
from llm_classifier import (
    load_classifiers, classify_all,
    show_examples, compute_llm_metrics, print_metrics_table
)

# Carga
classifiers = load_classifiers()

# 3 ejemplos cualitativos
ejemplos = [
    "El banco ofrece una cuenta de ahorros con interés anual del 3%.",
    "La tasa de rentabilidad ajustada por riesgo refleja la volatilidad "
    "implícita del subyacente en el mercado de derivados.",
    "Puedes guardar tu dinero en el banco y ganar un pequeño porcentaje extra.",
]
show_examples(classifiers, ejemplos)

# Dataset completo — all_texts y all_labels vienen de load_feina()
predictions = classify_all(classifiers, all_texts)
metrics     = compute_llm_metrics(predictions, all_labels)
print_metrics_table(metrics)

### Comentarios — Ejemplos cualitativos

| # | Texto | Selectra | BART | ¿Correcto? |
|---|-------|----------|------|------------|
| 1 | "El banco ofrece una cuenta de ahorros..." | simple (0.69) | complex (0.67) | Selectra ✓ / BART ✗ |
| 2 | "La tasa de rentabilidad ajustada por riesgo..." | complejo (0.91) | complex (0.87) | Ambos ✓ |
| 3 | "Puedes guardar tu dinero en el banco..." | simple (0.79) | simple (0.72) | Ambos ✓ |

**Ejemplo 1:** Ambos modelos muestran baja confianza (~0.69 y ~0.67), lo que
indica que el texto está en una zona fronteriza. Selectra lo clasifica
correctamente como simple, mientras que BART lo etiqueta como complejo,
posiblemente porque procesa español como lengua extranjera y el vocabulario
financiero ("interés anual", "cuenta de ahorros") activa señales de
complejidad.

**Ejemplo 2:** Es el caso más claro — terminología técnica densa
("rentabilidad ajustada por riesgo", "volatilidad implícita", "subyacente",
"derivados") es reconocida como compleja por ambos modelos con alta
confianza (0.91 y 0.87). La coincidencia sugiere que este tipo de
complejidad léxica es robusta incluso entre idiomas.

**Ejemplo 3:** Ambos modelos coinciden en `simple` con confianza moderada.
Esto es esperable dado que el texto usa vocabulario cotidiano y estructura
sintáctica sencilla, a pesar de referirse al mismo tema financiero que el
ejemplo 1 — lo que indica que ambos modelos son sensibles a la complejidad
léxica y no solo al dominio temático.

**Observación general:** Selectra, al estar entrenado en español, muestra
mayor confianza promedio y mejor criterio en casos ambiguos. BART, siendo
un modelo en inglés, puede perder matices semánticos del español, lo que
se refleja en su error en el ejemplo 1.

### Comentarios — Resultados sobre el dataset completo

| Modelo    | Accuracy | MAE    |
|-----------|----------|--------|
| Selectra  | 0.5217   | 0.4783 |
| BART      | 0.4995   | 0.5005 |

Los resultados son sorprendentes y requieren interpretación cuidadosa:

**Rendimiento cercano al azar:** Ambos modelos rondan el 50% de accuracy,
que es exactamente lo que obtendría un clasificador aleatorio en un dataset
balanceado. Esto contrasta con el buen desempeño observado en los 3 ejemplos
cualitativos, donde ambos modelos mostraban confianza razonable.

**BART por debajo del azar:** Con accuracy de 0.4995 y MAE de 0.5005, BART
está esencialmente invirtiendo las etiquetas — clasifica como `complex` lo
que es simple y viceversa. Esto sugiere que sus labels en inglés
("simple" / "complex") no se alinean bien con las etiquetas del dataset
cuando se procesa texto en español a escala.

**Selectra apenas supera el azar:** A pesar de estar entrenado en español,
0.5217 es un margen marginal sobre el 50%. Esto indica que la tarea de
clasificación del dataset FEINA no se alinea bien con los labels
"simple" / "complejo" usados en el zero-shot, o que el criterio de
complejidad del dataset difiere del que el modelo asume intrínsecamente.

**Causa probable:** Ambos modelos son zero-shot, es decir, no fueron
entrenados específicamente para esta tarea. La definición de "complejo"
en FEINA (simplificación lexical de textos) puede diferir significativamente
de lo que estos modelos asocian con complejidad textual general.

**Conclusión:** Para esta tarea específica, los LLMs zero-shot no superan
una línea base aleatoria, lo que refuerza el valor de los enfoques
supervisados como la regresión logística con TF-IDF o embeddings de BERT,
que tienen acceso directo a las etiquetas del dataset durante el
entrenamiento.

# Sección 2.4 Modelos LLM con few shot

In [ ]:
# Few-Shot LLMs
from few_shot_classifier import (
    load_few_shot_classifiers, classify_all_configs,
    show_few_shot_examples, compute_few_shot_metrics,
    print_few_shot_metrics
)

# Carga (reutiliza los mismos modelos que la sección anterior)
classifiers = load_few_shot_classifiers()

# 3 ejemplos cualitativos
ejemplos = [
    "El banco ofrece una cuenta de ahorros con interés anual del 3%.",
    "La tasa de rentabilidad ajustada por riesgo refleja la volatilidad "
    "implícita del subyacente en el mercado de derivados.",
    "Puedes guardar tu dinero en el banco y ganar un pequeño porcentaje extra.",
]
show_few_shot_examples(classifiers, ejemplos)

# Dataset completo
results = classify_all_configs(classifiers, all_texts)
metrics = compute_few_shot_metrics(results, all_labels)
print_few_shot_metrics(metrics)